### 1. Config & Environment Setup


In [ ]:
import os, sys, time, re, json, random, warnings
from collections import Counter
from pathlib import Path

print("⚙️  Installing dependencies...")

    if any(p in mod for p in ['numpy','scipy','torch','transformers']):

import json, random, re, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from pathlib import Path
from collections import Counter, defaultdict
from datasets import Dataset
import evaluate
from transformers import (
    MT5ForConditionalGeneration, Seq2SeqTrainer, Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq, EarlyStoppingCallback, AutoTokenizer, AutoModelForSequenceClassification
)
from transformers import T5TokenizerFast as MT5Tokenizer
from rank_bm25 import BM25Okapi
from rouge_score import rouge_scorer as rs_module

warnings.filterwarnings('ignore')
SEED = 42
random.seed(SEED); np.random.seed(SEED)

# ── TIMER ─────────────────────────────────────────────────────────────────────
PIPELINE_START  = time.time()
WALL_LIMIT_SECS = 60 * 60  # Strict 1-hour run limit

def elapsed():   return time.time() - PIPELINE_START
def time_left(): return WALL_LIMIT_SECS - elapsed()

def check_time(label=''):
    e, r = elapsed(), time_left()
    print(f'  ⏱  {label} | elapsed={e/60:.1f}min | remaining={r/60:.1f}min')
    return r > 180

# ── PATHS ─────────────────────────────────────────────────────────────────────
BASE               = 'DATA_DIR'
TEST_PATH          = f'{BASE}/Test.csv'
WORK               = '/kaggle/working'
OPTIMIZED_TRAIN_PATH = f'{BASE}/optimized_train.csv'
OPTIMIZED_VAL_PATH   = f'{BASE}/optimized_val.csv'
RAW_TRAIN_PATH     = f'{BASE}/Train.csv'
RAW_VAL_PATH       = f'{BASE}/Val.csv'
CONFIGS_JSON       = f'{WORK}/rag_configs.json'
STATS_CSV          = f'{WORK}/dialect_stats.csv'
QA_LOG_CSV         = f'{WORK}/qa_log.csv'
SUBMISSION         = f'{WORK}/submission_final.csv'
MODEL_DIR          = f'{WORK}/mt5_model'

LENGTH_CFG = {
    "Amh_Eth": {"max": 80,  "min": 10}, "Eng_Eth": {"max": 90,  "min": 15},
    "Eng_Gha": {"max": 220, "min": 20}, "Eng_Ken": {"max": 220, "min": 20},
    "Eng_Uga": {"max": 260, "min": 20}, "Lug_Uga": {"max": 220, "min": 20},
    "Swa_Ken": {"max": 230, "min": 20}, "Aka_Gha": {"max": 270, "min": 30},
}

OFFLINE_RAG_CONFIGS = {
    "Eng_Uga": {"analyzer": "word", "k1": 1.2, "b": 0.75, "copy_thresh": 0.85, "strip_prefix": True},
    "Swa_Ken": {"analyzer": "word", "k1": 1.2, "b": 0.75, "copy_thresh": 0.65, "strip_punct": True},
    "Eng_Eth": {"analyzer": "word", "k1": 1.2, "b": 0.75, "copy_thresh": 0.65},
    "Eng_Ken": {"analyzer": "word", "k1": 1.2, "b": 0.75, "copy_thresh": 0.45, "normalize_hiv_aids": False},
    "Lug_Uga": {"analyzer": "word", "k1": 1.2, "b": 0.75, "copy_thresh": 0.45, "strip_lug_prefix": True, "strip_punct": True},
    "Amh_Eth": {"analyzer": "word", "k1": 1.8, "b": 0.4, "copy_thresh": 0.65, "term_override": True, "search_answers": False},
    "Eng_Gha": {"analyzer": "word", "k1": 1.8, "b": 0.4, "copy_thresh": 0.45, "vocab_boost": True},
    "Aka_Gha": {"analyzer": "char_wb", "ngram": [2, 3], "k1": 1.5, "b": 0.75, "copy_thresh": 0.45, "strip_punct": True, "decompose": False},
}


### 2. Data Preparation


In [ ]:
# ── STAGE 1: DATA PREPARATION ─────────────────────────────────────────────────
def stage1_data_prep():
    print('\n' + '='*60); print('STAGE 1: Data preparation + Leakage purification'); print('='*60)
    train_raw, val_raw, test_df = pd.read_csv(OPTIMIZED_TRAIN_PATH), pd.read_csv(OPTIMIZED_VAL_PATH), pd.read_csv(TEST_PATH)
    
    for df in (train_raw, val_raw, test_df):
        for col in ('input', 'output'):
            if col in df.columns: df[col] = df[col].astype(str).str.replace(r'\s+', ' ', regex=True).str.strip()

    def make_key(df): return df['subset'] + '|||' + df['input'].str.lower().str.strip()
    test_keys, val_keys = set(make_key(test_df)), set(make_key(val_raw))

    train_raw = train_raw[~make_key(train_raw).isin(test_keys)].copy()
    train_raw = train_raw[~make_key(train_raw).isin(val_keys)].copy()
    train_raw['_key'] = make_key(train_raw)
    train_raw = train_raw.drop_duplicates('_key').drop(columns='_key')
    
    val_raw['_key'] = make_key(val_raw)
    val_raw = val_raw.drop_duplicates('_key').drop(columns='_key')
    val_raw = val_raw[~make_key(val_raw).isin(test_keys)].copy()

    from sklearn.model_selection import train_test_split
    val_to_train, val_held = train_test_split(val_raw, test_size=0.10, stratify=val_raw['subset'], random_state=SEED)
    full_train = pd.concat([train_raw, val_to_train], ignore_index=True).sample(frac=1, random_state=SEED).reset_index(drop=True)
    
    # Stratified subset sampling limits to bound training runtime strictly within 1 hour
    TARGET = {'Amh_Eth':0.10,'Aka_Gha':0.15,'Swa_Ken':0.10,'Lug_Uga':0.12,'Eng_Eth':0.12,'Eng_Gha':0.12,'Eng_Ken':0.09,'Eng_Uga':0.20}
    total, frames = len(full_train), []
    for subset, ratio in TARGET.items():
        sub = full_train[full_train['subset'] == subset]
        n_target = min(int(total * ratio), int(len(sub) * 1.20))
        frames.append(sub.sample(n_target, replace=True, random_state=SEED) if n_target > len(sub) else sub.sample(n_target, random_state=SEED))

    full_train = pd.concat(frames).sample(frac=1, random_state=SEED).reset_index(drop=True)
    check_time('Stage 1 complete')
    return full_train, val_held, test_df

# ── SCORING ───────────────────────────────────────────────────────────────────
_ROUGE = rs_module.RougeScorer(['rouge1', 'rougeL'], use_stemmer=False)

def score_pair(pred, ref, subset=None):
    try:
        s = _ROUGE.score(str(ref), str(pred))
        return {'rouge1': round(s['rouge1'].fmeasure,4), 'rougeL': round(s['rougeL'].fmeasure,4), 'blended': round(.5*s['rouge1'].fmeasure+.5*s['rougeL'].fmeasure,4)}
    except Exception: return {'rouge1': 0., 'rougeL': 0., 'blended': 0.}

def score_batch(preds, refs, subsets): return pd.DataFrame([score_pair(p, r, s) for p, r, s in zip(preds, refs, subsets)])


### 3. Retrieval & RAG Architecture


In [ ]:
# ── RAG + CROSS-ENCODER ARCHITECTURE ──────────────────────────────────────────
AMHARIC_OVERRIDES = {'ቂጥኝ':'syphilis','የቂጥኝ':'syphilis','ኪንታሮት':'genital warts','ክላሚዲያ':'chlamydia','ሄርፒስ':'herpes','ኤችፒቪ':'HPV','ኤችአይቪ':'HIV','ኤድስ':'AIDS'}
_LUG_PREFIXES = sorted(['aba','obu','eki','ebi','oli','olu','aka','aga','omu','emi','oku','nga','ba','bi','bu','ko','mu','ku','wa','e','o','n'], key=len, reverse=True)

def _strip_lug(word):
    for p in _LUG_PREFIXES:
        if word.startswith(p) and len(word) > len(p)+3: return word[len(p):]
    return word

def make_tokenizer(subset, cfg):
    if subset == 'Amh_Eth': return lambda t: [w for w in re.sub(r'[፡።፣፤፥፦\u1361-\u1368\s]+', ' ', str(t)).strip().split() if w]
    analyzer, ngram, sp, slp = cfg.get('analyzer', 'word'), cfg.get('ngram', (1, 1)), cfg.get('strip_punct', True), cfg.get('strip_lug_prefix', False) and subset == 'Lug_Uga'
    def tok(text):
        t = str(text).lower()
        if sp: t = re.sub(r'[?!.,;:\'\"\-]+', ' ', t)
        t = re.sub(r'\s+', ' ', t).strip()
        if analyzer == 'char_wb':
            lo, hi = ngram; tokens = []
            for w in t.split():
                wp = f' {w} '
                for n in range(lo, hi+1): tokens += [wp[i:i+n] for i in range(len(wp)-n+1)]
            return tokens
        words = [w for w in t.split() if len(w) > 2]
        if slp: words = [_strip_lug(w) for w in words]
        return words
    return tok

def build_bm25(texts, tok): return BM25Okapi([tok(t) for t in texts], k1=1.5, b=0.75)

def preprocess(text, subset, cfg):
    t = re.sub(r'\s+', ' ', str(text)).strip()
    if subset == 'Amh_Eth' and cfg.get('term_override', True):
        for k, v in AMHARIC_OVERRIDES.items(): t = t.replace(k, v)
        return t
    if subset in ('Eng_Ken','Eng_Gha','Eng_Uga','Eng_Eth'): t = re.sub(r'\bHIV/AIDS\b', 'HIV', t, flags=re.I)
    if subset == 'Eng_Uga' and cfg.get('strip_prefix'): t = re.sub(r'^please\s+answer\s+using\s+simple\s+medical\s+terms[^.]*\.\s*', '', t, flags=re.I).strip()
    return t

# ══════════════════════════════════════════════════════════════════════════════
# CROSS-ENCODER RE-RANKING INFRASTRUCTURE
# ══════════════════════════════════════════════════════════════════════════════
class CrossEncoderReranker:
    def __init__(self):
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        # SOTA highly lightweight multilingual cross-encoder backbone
        self.model_name = 'BAAI/bge-reranker-base'
        print(f"🚀 Initializing Neural Cross-Encoder Layer: {self.model_name}")
        self.tokenizer = AutoTokenizer.from_pretrained(self.model_name)
        self.model = AutoModelForSequenceClassification.from_pretrained(self.model_name).to(self.device)
        if torch.cuda.is_available(): self.model = self.model.half()
        self.model.eval()

    def rerank(self, query, candidates, top_n=3):
        if not candidates: return [], []
        pairs = [[query, c] for c in candidates]
        inputs = self.tokenizer(pairs, padding=True, truncation=True, max_length=256, return_tensors='pt').to(self.device)
        
        with torch.no_grad():
            scores = self.model(**inputs).logits.view(-1).float()
            # Sigmoidal normalization of ranking confidence vectors
            probs = torch.sigmoid(scores).cpu().numpy().tolist()
            
        sorted_indices = np.argsort(probs)[::-1]
        return [candidates[i] for i in sorted_indices[:top_n]], [probs[i] for i in sorted_indices[:top_n]]

# ── FALLBACK MAPS ─────────────────────────────────────────────────────────────
FALLBACK_T = {
    'Aka_Gha': "Ɛsɛ sɛ wokɔ kliniki a ɛbɛn wo hɔ na wofa twerɛbea ne nhyehyɛe a ɛfata.",
    'Lug_Uga': "Genda ku ddwaliro eby'omulamwa okufuna obuyambi n'ebiragiro.",
    'Swa_Ken': "Tafadhali wasiliana na kituo cha afya karibu nawe kupata ushauri na matibabu.",
    'Eng_Uga': "Please visit your nearest health facility for advice and treatment.",
    'Eng_Ken': "Please seek medical attention at your nearest health facility.",
    'Eng_Eth': "Visit a health facility for testing and treatment.",
}

def get_fallback(subset): return FALLBACK_T.get(subset, 'Please visit your nearest health facility.')


### 4. Model Training


In [ ]:
# ── STAGE 3: TRAINING BACKBONE WITH STABILIZED CROSS-ENTROPY MASKING ──────────
def stage3_train_model(train_df, val_df):
    print('\n' + '='*60); print('STAGE 3: mT5-Large stabilized cross-entropy training loop'); print('='*60)
    if not check_time('Stage 3 sequence initialized'): return None, None

    try:
        MODEL = 'google/mt5-large'
        tokenizer = MT5Tokenizer.from_pretrained(MODEL)

        def tokenize(batch):
            inp = tokenizer(batch['input'], max_length=128, truncation=True, padding=False)
            lbl = tokenizer(text_target=batch['output'], max_length=128, truncation=True, padding=False)
            # FIX: Replace pad tokens explicitly with -100 to decouple cross-entropy tracking from structural sequence padding length
            inp['labels'] = [[(t if t != tokenizer.pad_token_id else -100) for t in label] for label in lbl['input_ids']]
            return inp

        # Safe bounding setup for a fast 1-hour total pipeline step tracking run
        train_ds = Dataset.from_pandas(train_df[['input','output']].dropna()).map(tokenize, batched=True, remove_columns=['input','output'])
        val_ds   = Dataset.from_pandas(val_df[['input','output']].dropna()).map(tokenize, batched=True, remove_columns=['input','output']).select(range(min(100, len(val_df))))

        model = MT5ForConditionalGeneration.from_pretrained(MODEL, use_safetensors=True)
        if hasattr(model, "gradient_checkpointing_enable"): model.gradient_checkpointing_enable()

        args = Seq2SeqTrainingArguments(
            output_dir=MODEL_DIR, max_steps=120, per_device_train_batch_size=2, per_device_eval_batch_size=1,
            fp16=torch.cuda.is_available(), logging_steps=30, optim='adafactor', learning_rate=5e-4,
            gradient_accumulation_steps=4, report_to='none'
        )

        trainer = Seq2SeqTrainer(
            model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds,
            processing_class=tokenizer, data_collator=DataCollatorForSeq2Seq(tokenizer, model=model, label_pad_token_id=-100)
        )

        trainer.train()
        return model, tokenizer
    except Exception as e:
        print(f'Training run bypassed or completed: {e}'); return None, None

def build_rag_indexes(corpus_df, best_configs):
    indexes = {}
    for subset in corpus_df['subset'].unique():
        cfg = best_configs.get(subset, {})
        sub = corpus_df[corpus_df['subset'] == subset].reset_index(drop=True)
        tok = make_tokenizer(subset, cfg)
        idx_t = [preprocess(q, subset, cfg) for q in sub['input'].astype(str)]
        indexes[subset] = {'bm25': build_bm25(idx_t, tok), 'tok': tok, 'pool': sub['output'].astype(str).tolist(), 'inputs_pool': sub['input'].astype(str).tolist()}
    return indexes


### 5. Inference & Diagnostics


In [ ]:
# ── STAGE 4: TWO-PASS RETRIEVAL HYBRID INFERENCE ──────────────────────────────
def stage4_inference(test_df, corpus_df, best_configs, model, tokenizer, reranker):
    print('\n' + '='*60); print('STAGE 4: Two-pass (BM25 -> Neural Cross-Encoder) inference matching'); print('='*60)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    if model: model.eval(); model.to(device)

    indexes = build_rag_indexes(corpus_df, best_configs)
    predictions, sources, scores_log = [], [], []

    for subset in test_df['subset'].unique():
        sub_test = test_df[test_df['subset'] == subset]
        cfg = best_configs.get(subset, {})
        thresh = cfg.get('copy_thresh', 0.80)
        idx = indexes.get(subset, {})
        bm, tok_fn, answers_pool, inputs_pool = idx.get('bm25'), idx.get('tok'), idx.get('pool', []), idx.get('inputs_pool', [])

        print(f'  [{subset}] Executing neural two-pass matching | thresh threshold boundary={thresh:.2f}')

        for _, row in sub_test.iterrows():
            q_raw = str(row['input'])
            q_proc = preprocess(q_raw, subset, cfg)

            pred, src, final_sc = None, 'TEMPLATE', 0.0

            if bm and tok_fn and answers_pool:
                bm25_scores = bm.get_scores(tok_fn(q_proc))
                # Pass 1: Select top 15 candidates lexically
                top_candidates_idx = np.argsort(bm25_scores)[-15:][::-1]
                candidate_answers = [answers_pool[i] for i in top_candidates_idx if bm25_scores[i] > 0]
                
                if candidate_answers:
                    # Pass 2: Neural Cross-Encoder deep attention verification step
                    ranked_ans, neural_scores = reranker.rerank(q_raw, candidate_answers, top_n=1)
                    if neural_scores:
                        final_sc = neural_scores[0]
                        if final_sc >= thresh:
                            pred, src = ranked_ans[0], 'RAG_CROSS_ENCODER'

            if not pred:
                if model and tokenizer:
                    inputs = tokenizer(f"answer health: {q_raw}", return_tensors='pt', max_length=128, truncation=True).to(device)
                    with torch.no_grad(): out = model.generate(**inputs, max_new_tokens=128)
                    pred = tokenizer.decode(out[0], skip_special_tokens=True).strip()
                    src = 'MODEL'
                else:
                    pred = get_fallback(subset)

            predictions.append(pred); sources.append(src)
            scores_log.append({'subset': subset, 'src': src, 'rag_score': final_sc})
            
    return predictions, pd.DataFrame(scores_log)

# ── STAGE 5: UNBIASED PERFORMANCE DIAGNOSTICS LOGGING ────────────────────────
def stage5_stats_and_submit(test_df, predictions, val_df, train_rag_clean, best_configs, model, tokenizer, reranker):
    print('\n' + '='*60); print('STAGE 5: Unbiased pipeline analysis metrics writeout'); print('='*60)
    pd.DataFrame({"ID": test_df["ID"], "TargetRLF1": predictions, "TargetR1F1": predictions, "TargetLLM": predictions}).to_csv(SUBMISSION, index=False)

    indexes = build_rag_indexes(train_rag_clean, best_configs)
    stat_rows, all_qa_rows = [], []

    for subset in sorted(val_df['subset'].unique()):
        sv = val_df[val_df['subset'] == subset].reset_index(drop=True)
        if len(sv) == 0: continue
        cfg = best_configs.get(subset, {})
        idx = indexes.get(subset, {})
        bm, tok_fn, answers_pool = idx.get('bm25'), idx.get('tok'), idx.get('pool', [])
        thresh = cfg.get('copy_thresh', 0.80)

        val_preds, val_srcs, val_rag_scores = [], [], []
        for _, row in sv.iterrows():
            q_raw, q_proc = str(row['input']), preprocess(str(row['input']), subset, cfg)
            pred, src, final_sc = None, 'TEMPLATE', 0.0

            if bm and tok_fn and answers_pool:
                bm25_scores = bm.get_scores(tok_fn(q_proc))
                top_candidates_idx = np.argsort(bm25_scores)[-15:][::-1]
                candidate_answers = [answers_pool[i] for i in top_candidates_idx if bm25_scores[i] > 0]
                
                if candidate_answers:
                    ranked_ans, neural_scores = reranker.rerank(q_raw, candidate_answers, top_n=1)
                    if neural_scores:
                        final_sc = neural_scores[0]
                        if final_sc >= thresh: pred, src = ranked_ans[0], 'RAG_CROSS_ENCODER'

            if not pred: pred = get_fallback(subset)
            val_preds.append(pred); val_srcs.append(src); val_rag_scores.append(final_sc)

        sc_df = score_batch(val_preds, sv['output'].tolist(), [subset]*len(sv))
        sc_df['q'], sc_df['pred'], sc_df['ref'], sc_df['src'], sc_df['rag_score'] = sv['input'].values, val_preds, sv['output'].values, val_srcs, val_rag_scores

        r1, rL, blended = sc_df['rouge1'].mean(), sc_df['rougeL'].mean(), sc_df['blended'].mean()
        
        print(f"\n📈 Dialect Unbiased Metric Trace: [{subset}] | True Blended Val ROUGE: {blended:.4f} | Neural RAG Hit Rate: {(sc_df['src'] == 'RAG_CROSS_ENCODER').mean()*100:.1f}%")
        print(f"   Worst Error Trace Target Match:\n   -> Q: {sc_df.nsmallest(1, 'blended').iloc[0]['q'][:85]}\n   -> P: {sc_df.nsmallest(1, 'blended').iloc[0]['pred'][:85]}")

        stat_rows.append({'subset': subset, 'n_val': len(sv), 'rouge1_mean': round(r1,4), 'blended_mean': round(blended,4), 'timestamp': pd.Timestamp.now().isoformat()})
        all_qa_rows.append(sc_df)

    pd.DataFrame(stat_rows).to_csv(STATS_CSV, index=False); pd.concat(all_qa_rows, ignore_index=True).to_csv(QA_LOG_CSV, index=False)
    print(f'\n✓ Diagnostics logged seamlessly across: {STATS_CSV} and {QA_LOG_CSV}')

# ── RUN CONTROL EXECUTION ─────────────────────────────────────────────────────
if __name__ == '__main__':
    train_df, val_df, test_df = stage1_data_prep()

    raw_train = pd.read_csv(RAW_TRAIN_PATH)
    for col in ('input','output'): raw_train[col] = raw_train[col].astype(str).str.replace(r'\s+',' ',regex=True).str.strip()
    
    def make_key(df): return df['subset'] + '|||' + df['input'].str.lower().str.strip()
    train_rag_clean = raw_train[~make_key(raw_train).isin(set(make_key(test_df)))].copy()

    best_configs = OFFLINE_RAG_CONFIGS
    best_configs["Aka_Gha"]["copy_thresh"] = 0.45

    with open(CONFIGS_JSON, 'w', encoding='utf-8') as f: json.dump(best_configs, f, indent=2, ensure_ascii=False)

    reranker = CrossEncoderReranker()
    model, tokenizer = stage3_train_model(train_df, val_df)
    
    predictions, scores_log = stage4_inference(test_df, train_rag_clean, best_configs, m
